# Importación de Librerías

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks, optimizers
from tensorflow.keras.utils import to_categorical
import numpy as np

# 1. Carga y Procesamiento
- Normalización / 255.0.
- One-hot encoding en 'y' para métrica AUC.
- Split: 5k val, 45k train.

In [ ]:
(X_train_full, y_train_full), (X_test, y_test) = keras.datasets.cifar10.load_data()

# Escalamos pixeles a rango [0, 1] para estabilidad numérica
X_train_full = X_train_full.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

# One-Hot Encoding para compatibilidad con métrica AUC
y_train_full = to_categorical(y_train_full, 10)
y_test = to_categorical(y_test, 10)

# Split: 5000 para validación, 45000 para entrenamiento
X_val, X_train = X_train_full[:5000], X_train_full[5000:]
y_val, y_train = y_train_full[:5000], y_train_full[5000:]

# 2. Construcción de la Red Profunda
- 20 capas ocultas, 100 neuronas c/u.
- Inicialización He: evita vanishing gradient.
- Batch Normalization: estabiliza, va antes de activación.
- Swish: activación.
- Salida: softmax (10 clases).

In [ ]:
model = models.Sequential()

model.add(layers.Flatten(input_shape=[32, 32, 3]))

for i in range(20):
    model.add(layers.Dense(
        100,
        kernel_initializer="he_normal"
    ))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation("swish"))

model.add(layers.Dense(10, activation="softmax"))

model.summary()

# 3. Configuración del Optimizador y Pérdida
- AdamW: regularización L2.
- Pérdida: categorical_crossentropy (por usar one-hot).
- Métricas: accuracy y AUC.

In [ ]:
optimizer = optimizers.AdamW(learning_rate=0.001, weight_decay=0.01)

model.compile(
    loss="categorical_crossentropy",
    optimizer=optimizer,
    metrics=[
        "accuracy",
        keras.metrics.AUC(name="auc")
    ]
)

# 4. Configuración de Callbacks (Regularización y Scheduling)
- Early Stopping: detiene si no hay mejora en 10 epochs. Guarda mejores pesos.
- ReduceLROnPlateau: divide LR a la mitad tras 5 epochs sin mejora.

In [ ]:
early_stopping_cb = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

lr_scheduler_cb = callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    verbose=1
)

# 5. Entrenamiento del Modelo
- 50 epochs, batch size 64.

In [ ]:
epochs = 50
batch_size = 64

history = model.fit(
    X_train, y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping_cb, lr_scheduler_cb]
)

# 6. Evaluación Final
- Se evalúa con el test set.

In [ ]:
eval_results = model.evaluate(X_test, y_test, verbose=1)

print(f"\nAccuracy final en Test: {eval_results[1]:.4f}")
print(f"AUC final en Test: {eval_results[2]:.4f}")